# AVX integrator

It appears that the integration routine requires the most time in the MPC problem.
This is because many small computations need to be performed sequentially.
However, many of these sequential computations can be performed in parallel, or in parallel with some time delay.
JAX probably has difficulty finding (and exploiting) this structure in a general program, so extra care probably needs to be taken with the programmer.
The following attempts this, in part by forming a giant (block) iteration matrix.

FYI: by AVX, I'm referencing Intel's [AVX](https://en.wikipedia.org/wiki/Advanced_Vector_Extensions).
However, I really mean architecture specific SIMD instructions, e.g., [ARM NEON][https://www.arm.com/technologies/neon].
I'll see if this improves performance...

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)


In [ ]:
import jax.numpy as jnp
import numpy as np
import scipy.linalg as sci_lin

import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.comp as comp

In [ ]:
n = 400
rstate_num = 12
vstate_num = 3 * const.E0_acc.shape[0] + 3 * const.E0_omega.shape[0]
control_num = 6

acc_ref = jnp.array([0.5, -0.2, 0.1]) + const.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.2, 0.1, -0.3])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

rstate0 = jnp.zeros(rstate_num)
vstate0_irl = jnp.zeros(vstate_num)
vstate0_sim = jnp.zeros(vstate_num)
control0 = jnp.zeros(control_num)

In [ ]:
np.random.seed(42)
control = utils.Control(jnp.array(np.random.uniform(-1.0, 1.0, control_num * n)).reshape(-1, control_num))

## initial benchmarking

This shouldn't involve many SIMD computations.

In [ ]:
get_rstate = utils.get_rstate
get_vstate_irl = utils.get_vstate_irl
get_vstate = utils.get_vstate

# get_rstate = jax.jit(utils.get_rstate)
# get_vstate_irl = jax.jit(utils.get_vstate_irl)
# get_vstate = jax.jit(utils.get_vstate)

@jax.jit
def zero_vstate() -> utils.VState:
    return utils.VState(jnp.zeros(vstate_num), jnp.zeros(6))

@jax.jit
def get_states(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control: utils.Control,
) -> tuple[utils.RState, utils.VState, utils.VState]:
    rstate = get_rstate(control, rstate0)

    vstate_irl = get_vstate_irl(rstate, control, control0, vstate0_irl)
    vstate_sim = get_vstate(acc_ref, omega_ref, vstate0_sim)

    # vstate_irl = zero_vstate()
    # vstate_sim = zero_vstate()

    return rstate, vstate_irl, vstate_sim

def get_state_call() -> tuple[utils.RState, utils.VState, utils.VState]:
    res = get_states(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=control,
    )
    res[0].state.block_until_ready()
    res[1].x_state.block_until_ready()
    res[1].y_state.block_until_ready()
    res[2].x_state.block_until_ready()
    res[2].y_state.block_until_ready()
    return res

In [ ]:
get_state_call()
with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
    get_state_call()

In [ ]:
get_state_call()  # compile
%timeit get_state_call()

## monolithic numpy loop

Implement `get_rstate`, `get_vstate_irl`, and `get_vstate` in one for monolithic for loop, hopefully for better SIMD performance?
First, we implement a numpy version for easier debugging.

In [ ]:
def monolithic_state_np(
    acc_ref: np.ndarray,
    omega_ref: np.ndarray,
    rstate0: np.ndarray,
    vstate0_irl: np.ndarray,
    vstate0_sim: np.ndarray,
    control0: np.ndarray,
    control: np.ndarray,
) -> tuple[utils.RState, utils.VState, utils.VState]:
    # the more subtle assertions...
    assert acc_ref.shape == omega_ref.shape
    assert acc_ref.shape[0] == control.shape[0]
    assert control0.shape == (control.shape[1],)
    assert rstate0.shape == (12,)
    assert vstate0_irl.shape == (15,)
    assert vstate0_sim.shape == (15,)

    n = acc_ref.shape[0]  # horizon num

    x0 = np.concatenate([rstate0, vstate0_irl, vstate0_sim])
    x_res = np.empty((n + 1, x0.size))
    x_res[0] = x0

    # append initial control, which is only applied to vstate_irl
    control = np.vstack([control0.reshape(1, -1), control])

    def comp_uv_irl(x: np.ndarray, u: np.ndarray) -> np.ndarray:
        assert x.shape == (42,)
        assert u.shape == (6,)

        xr = x[:12]

        euler = xr[3: 6]
        euler_dot = xr[9: 12]
        R = comp.rot(*euler)

        u_acc = np.ravel(R.T @ (u[:3] + const.gravity))
        u_omega = np.ravel(comp.angle_vel(*np.concatenate([euler, euler_dot])))

        return np.concatenate([u_acc, u_omega])

    for i in range(1, x_res.shape[0]):
        ur = control[i]
        uv_irl = comp_uv_irl(x_res[i - 1], control[i - 1])
        uv_sim = np.concatenate([acc_ref[i - 1], omega_ref[i - 1]])
        u = np.concatenate([ur, uv_irl, uv_sim])
        assert u.shape == (18,)

        x_res[i] = const.E0 @ x_res[i - 1] + const.E1 @ u

    xr = x_res[:, :12]
    xv_sim = x_res[:, 12 + 15:]

    xv_irl = np.empty((n + 1, 15))
    xv_irl[:-1, :] = x_res[1:, 12: 12 + 15]
    uv_irl = comp_uv_irl(x_res[-1], control[-1])
    xv_irl[-1] = const.E0_ves @ xv_irl[-2] + const.E1_ves @ uv_irl

    yv_irl = np.transpose(const.C_ves @ xv_irl.T)
    yv_sim = np.transpose(const.C_ves @ xv_sim.T)

    rstate = utils.RState(jnp.array(xr))
    vstate_irl = utils.VState(jnp.array(xv_irl), jnp.array(yv_irl))
    vstate_sim = utils.VState(jnp.array(xv_sim), jnp.array(yv_sim))
    return rstate, vstate_irl, vstate_sim

def get_monolithic_state_call_np() -> tuple[utils.RState, utils.VState, utils.VState]:
    res = monolithic_state_np(
        acc_ref=np.array(acc_ref),
        omega_ref=np.array(omega_ref),
        rstate0=np.array(rstate0),
        vstate0_irl=np.array(vstate0_irl),
        vstate0_sim=np.array(vstate0_sim),
        control0=np.array(control0),
        control=np.array(control.control),
    )
    res[0].state.block_until_ready()
    res[1].x_state.block_until_ready()
    res[1].y_state.block_until_ready()
    res[2].x_state.block_until_ready()
    res[2].y_state.block_until_ready()
    return res

In [ ]:
check0 = get_state_call()
check1 = get_monolithic_state_call_np()
max(
    np.max(np.abs(check0[0].state - check1[0].state)),
    np.max(np.abs(check0[1].x_state - check1[1].x_state)),
    np.max(np.abs(check0[1].y_state - check1[1].y_state)),
    np.max(np.abs(check0[2].x_state - check1[2].x_state)),
    np.max(np.abs(check0[2].y_state - check1[2].y_state)),
)

In [ ]:
%timeit get_monolithic_state_call_np()

## monolithic jax loop

In [ ]:
@jax.jit
def monolithic_state(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control: utils.Control,
) -> tuple[utils.RState, utils.VState, utils.VState]:
    n = acc_ref.shape[0]  # horizon num

    x0 = jnp.concatenate([rstate0, vstate0_irl, vstate0_sim])
    x_res = jnp.empty((n + 1, x0.size))
    x_res = x_res.at[0].set(x0)

    # append initial control, which is only applied to vstate_irl
    control_arr = jnp.vstack([control0.reshape(1, -1), control.control])

    def comp_uv_irl(x: jax.Array, u: jax.Array) -> jax.Array:
        assert x.shape == (42,)
        assert u.shape == (6,)

        xr = x[:12]

        euler = xr[3: 6]
        euler_dot = xr[9: 12]
        euler_all = jnp.concatenate([euler, euler_dot])
        R = comp.rot(*euler)

        u_acc = jnp.ravel(R.T @ (u[:3] + const.gravity))
        u_omega = jnp.ravel(comp.angle_vel(*euler_all))
        return jnp.concatenate([u_acc, u_omega])

    # for i in range(1, x_res.shape[0]):
    def for_body(i: int, x_res: jax.Array) -> jax.Array:
        ur = control_arr[i]
        uv_irl = comp_uv_irl(x_res[i - 1], control_arr[i - 1])
        uv_sim = jnp.concatenate([acc_ref[i - 1], omega_ref[i - 1]])
        u = jnp.concatenate([ur, uv_irl, uv_sim])
        assert u.shape == (18,)

        x_res_i = const.E0 @ x_res[i - 1] + const.E1 @ u
        return x_res.at[i].set(x_res_i)

    x_res = jax.lax.fori_loop(1, x_res.shape[0], for_body, x_res)

    xr = x_res[:, :12]
    xv_sim = x_res[:, 12 + 15:]

    xv_irl = jnp.empty((n + 1, 15))
    xv_irl = xv_irl.at[:-1, :].set(x_res[1:, 12: 12 + 15])
    uv_irl = comp_uv_irl(x_res[-1], control_arr[-1])
    xv_irl_1 = const.E0_ves @ xv_irl[-2] + const.E1_ves @ uv_irl
    xv_irl = xv_irl.at[-1].set(xv_irl_1)

    yv_irl = jnp.transpose(const.C_ves @ xv_irl.T)
    yv_sim = jnp.transpose(const.C_ves @ xv_sim.T)

    rstate = utils.RState(xr)
    vstate_irl = utils.VState(xv_irl, yv_irl)
    vstate_sim = utils.VState(xv_sim, yv_sim)
    return rstate, vstate_irl, vstate_sim

def get_monolithic_state_call() -> tuple[utils.RState, utils.VState, utils.VState]:
    res = monolithic_state(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=control,
    )
    res[0].state.block_until_ready()
    res[1].x_state.block_until_ready()
    res[1].y_state.block_until_ready()
    res[2].x_state.block_until_ready()
    res[2].y_state.block_until_ready()
    return res

In [ ]:
check0 = get_state_call()
check1 = get_monolithic_state_call()
max(
    np.max(np.abs(check0[0].state - check1[0].state)),
    np.max(np.abs(check0[1].x_state - check1[1].x_state)),
    np.max(np.abs(check0[1].y_state - check1[1].y_state)),
    np.max(np.abs(check0[2].x_state - check1[2].x_state)),
    np.max(np.abs(check0[2].y_state - check1[2].y_state)),
)

In [ ]:
# get_monolithic_state_call()
# with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
#     get_monolithic_state_call()

In [ ]:
get_monolithic_state_call()
%timeit get_monolithic_state_call()